# 09 · Webhooks — be told when a job finishes

**Use case:** a nightly pipeline re-uploads data and cleans; instead of polling, the orchestrator is told when the clean
finishes (`job.succeeded`) and kicks off the next step.

**Sub-tasks**
1. Register an https endpoint (here a throw-away https://webhook.site URL) for `clean` jobs
2. Send a signed test ping and verify the signature the way a receiver would
3. Run a clean and watch `job.succeeded` arrive exactly once
4. Read the delivery log; rotate the secret; delete the webhook

Set `WEBHOOK_URL` (and `WEBHOOK_SITE_TOKEN` — the uuid in the URL — to read what arrived) before running; without them
the notebook explains and skips the live steps.

In [ ]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

In [ ]:
import os, time, urllib.request
from langsat.webhooks import verify_signature

URL = os.environ.get("WEBHOOK_URL")
TOKEN = os.environ.get("WEBHOOK_SITE_TOKEN")
if not URL:
    print("WEBHOOK_URL not set — open https://webhook.site, copy 'Your unique URL', and set WEBHOOK_URL / WEBHOOK_SITE_TOKEN")
p = get_or_create_project(ls, "explore", kind="data_analysis")
from langsat import errors
try:
    print("webhooks enabled:", ls.webhooks.enabled()["enabled"], "· key scopes:", [s for s in ls.me()["api_key"]["scopes"] if s.startswith("webhooks")])
    ls.webhooks.list()
except errors.MissingScope as e:
    print(f"this key cannot manage webhooks ({e.scope} missing) — mint one with the Full SDK preset; skipping the live steps")
    URL = None

In [ ]:
def received():
    with urllib.request.urlopen(f"https://webhook.site/token/{TOKEN}/requests?sorting=newest", timeout=20) as r:
        return json.load(r)["data"]

if URL:
    for w in ls.webhooks.list():
        if (w.get("description") or "") == "amazon-reviews example":
            ls.webhooks.delete(w["id"])
    w = ls.webhooks.create(URL, job_kinds=["clean"], description="amazon-reviews example")
    secret = w["secret"]                       # shown once
    print("webhook", w["id"], "events", w["events"], "kinds", w["job_kinds"])
    t = ls.webhooks.test(w["id"])
    print("ping delivered:", t["ok"], "· HTTP", t["delivery"]["last_status_code"])
    if TOKEN:
        time.sleep(2)
        hit = next(x for x in received() if x["headers"].get("x-langsat-event") == ["ping"])
        ev = verify_signature(hit["content"], hit["headers"]["x-langsat-signature"][0], secret=secret)
        print("signature verified · event", ev["id"], ev["type"])

In [ ]:
if URL:
    p.cleaning.clean().wait(timeout=1800)
    print("clean finished; waiting for the webhook …")
    got = None
    if TOKEN:
        for _ in range(24):
            time.sleep(5)
            got = next((x for x in received() if x["headers"].get("x-langsat-event") == ["job.succeeded"]), None)
            if got: break
        if got:
            ev = verify_signature(got["content"], got["headers"]["x-langsat-signature"][0], secret=secret)
            job = ev["data"]["job"]
            print("job.succeeded ·", job["kind"], job["status"], "· project", ev["data"]["project_id"] == p.id, "· fetch the result at", job["result_route"])
            n = sum(1 for x in received() if x["headers"].get("x-langsat-event") == ["job.succeeded"])
            print("deliveries of job.succeeded:", n)
    for d in ls.webhooks.deliveries(w["id"]):
        print(f"  {d['event']:<14} {d['status']:<10} attempts={d['attempts']} http={d['last_status_code']}")

In [ ]:
if URL:
    rotated = ls.webhooks.rotate_secret(w["id"])
    print("rotated:", rotated["secret"] != secret)
    ls.webhooks.delete(w["id"]); print("deleted")
save_metrics(".", {"notebook": "09_webhooks", "task": "webhooks · job.succeeded on clean", "model": "—", "project_id": p.id,
                   "headline": {"ran_live": bool(URL), "ping_ok": (t["ok"] if URL else None), "job_succeeded_seen": (bool(got) if URL and TOKEN else None)}})